# imports 

In [11]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
import csv
import os
import numpy as np
import json
from sklearn.metrics import accuracy_score
from resurse import Bow, tfidf
from reteleNeuronale.AnnCode import AnnCode
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import neural_network

# Login Azure

In [12]:
'''
Authenticate
Authenticates your credentials and creates a client.
'''
credidential = json.load(open("credidentiale.json"))
subscription_key = credidential["B"]["API_KEY"]
endpoint = credidential["B"]["END_POINT"]
computervision_client = TextAnalyticsClient(endpoint=endpoint, credential=AzureKeyCredential(subscription_key))
'''
END - Authenticate
'''

'\nEND - Authenticate\n'

# Azure Sentiment Analysis

In [13]:
documents = [
    "By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."
]

result = computervision_client.analyze_sentiment(documents, show_opinion_mining=True)
docs = [doc for doc in result if not doc.is_error]

print("Let's visualize the sentiment of each of these documents")
for idx, doc in enumerate(docs):
    print(f"Document text: {documents[idx]}")
    print(f"Overall sentiment: {doc.sentiment}")

Let's visualize the sentiment of each of these documents
Document text: By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.
Overall sentiment: positive


# Extragere Caracteristici din Text

## Citire texte

In [14]:
def readTextData(fileName):
    data = []
    with open(fileName) as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=',')
        line_count = 0
        for row in csv_reader:
            if line_count == 0:
                dataNames = row
            else:
                data.append(row)
            line_count += 1
    
    inputs = [data[x][0] for x in range(len(data))]
    outputs = [data[x][1] for x in range(len(data))]
    labelNames = list(set(outputs))
    
    return inputs,outputs,labelNames

## Train/Test Data 

In [15]:
def splitData(inputs, outputs):
    np.random.seed(43)
    indexes = [i for i in range(len(inputs))]
    trainSample = np.random.choice(indexes, size=int(0.8*len(inputs)), replace=False)
    testSample = [i for i in indexes if i not in trainSample]
    
    trainInputs = [inputs[i] for i in trainSample]
    trainOutputs = [outputs[i] for i in trainSample]
    testInputs = [inputs[i] for i in testSample]
    testOutputs = [outputs[i] for i in testSample]
    
    return trainInputs, trainOutputs, testInputs, testOutputs

## Logic

In [16]:
inp,out,labels = readTextData(os.path.join(os.getcwd(),'textData','test.csv'))
print(inp[:10])

['I love cycling to work every day instead of driving my car.', 'Choosing to walk instead of taking the car has improved my health.', "Reducing our environmental footprint is everyone's responsibility.", 'I feel proud to be part of a bike-sharing movement in my city.', 'The car broke down again, making me late for the important meeting.', 'Environmental regulations are making it harder for small businesses to operate.', "I'm frustrated about constantly choosing between convenience and eco-friendly options.", 'Cycling in heavy traffic is dangerous and stressful.', 'The new eco-friendly transportation options in our city are amazing.', 'I hate how reducing my carbon footprint requires so much extra effort.']


## Bag of Words

In [17]:
inputBowSets = Bow.bow_datasets(inp)
voc = Bow.vocabulary(Bow.preprocess(inp))
print(voc)
print([sum(set) for set in inputBowSets])

{'love': 0, 'cycling': 1, 'work': 2, 'every': 3, 'day': 4, 'instead': 5, 'drive': 6, 'car': 7, 'choose': 8, 'walk': 9, 'take': 10, 'improve': 11, 'health': 12, 'reduce': 13, 'environmental': 14, 'footprint': 15, 'everyone': 16, 'responsibility': 17, 'feel': 18, 'proud': 19, 'part': 20, 'bike': 21, 'share': 22, 'movement': 23, 'city': 24, 'broke': 25, 'make': 26, 'late': 27, 'important': 28, 'meeting': 29, 'regulation': 30, 'harder': 31, 'small': 32, 'business': 33, 'operate': 34, 'frustrate': 35, 'constantly': 36, 'convenience': 37, 'eco': 38, 'friendly': 39, 'option': 40, 'heavy': 41, 'traffic': 42, 'dangerous': 43, 'stressful': 44, 'new': 45, 'transportation': 46, 'amaze': 47, 'hate': 48, 'carbon': 49, 'require': 50, 'much': 51, 'extra': 52, 'effort': 53, 'expression': 54, 'talk': 55, 'electric': 56, 'contagious': 57, 'impact': 58, 'fast': 59, 'fashion': 60, 'devastate': 61, 'planet': 62, 'promotes': 63, 'personal': 64, 'sustainability': 65, 'regret': 66, 'path': 67, 'today': 68, 'fi

## TF-IDF

In [18]:
inputTFIDF = tfidf.set_tf_idf(inp)
print(inputTFIDF[0])

[np.float64(0.6045352383689347), np.float64(0.2579616480889621), np.float64(0.4312484432289484), np.float64(0.6045352383689347), np.float64(0.46720870228542105), np.float64(0.46720870228542105), np.float64(0.4312484432289484), np.float64(0.18800967459703427), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float6

# Antrenare ANN tool

## BOW

In [19]:
inputs = inputBowSets.copy()
outputs = [ 1 if o == 'positive' else 0 for o in out]

TI,TO,VI,VO = splitData(inputs,outputs)

toolClassifier1 = neural_network.MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    max_iter=500,
    solver='sgd',
    verbose=10,
    learning_rate_init=0.0001,
)
toolClassifier1.fit(TI, TO)

predicted = toolClassifier1.predict(VI)
print("acc: ", accuracy_score(VO, predicted))

Iteration 1, loss = 0.69603668
Iteration 2, loss = 0.69602776
Iteration 3, loss = 0.69601505
Iteration 4, loss = 0.69599892
Iteration 5, loss = 0.69597971
Iteration 6, loss = 0.69595769
Iteration 7, loss = 0.69593317
Iteration 8, loss = 0.69590641
Iteration 9, loss = 0.69587764
Iteration 10, loss = 0.69584705
Iteration 11, loss = 0.69581485
Iteration 12, loss = 0.69578120
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
acc:  0.6538461538461539


## TF-IDF

In [20]:
inputs = inputTFIDF.copy()
outputs = [ 1 if o == 'positive' else 0 for o in out]

TI,TO,VI,VO = splitData(inputs,outputs)

toolClassifier2 = neural_network.MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    max_iter=500,
    solver='sgd',
    verbose=10,
    learning_rate_init=0.0001,
)
toolClassifier2.fit(TI, TO)

predicted = toolClassifier2.predict(VI)
print("acc: ", accuracy_score(VO, predicted))

Iteration 1, loss = 0.69409443
Iteration 2, loss = 0.69408888
Iteration 3, loss = 0.69408097
Iteration 4, loss = 0.69407092
Iteration 5, loss = 0.69405896
Iteration 6, loss = 0.69404529
Iteration 7, loss = 0.69403007
Iteration 8, loss = 0.69401347
Iteration 9, loss = 0.69399563
Iteration 10, loss = 0.69397668
Iteration 11, loss = 0.69395673
Iteration 12, loss = 0.69393586
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
acc:  0.5384615384615384


# Antrenare ANN cod propriu

## BOW

In [24]:
inputs = inputBowSets.copy()
outputs = [ 1 if o == 'positive' else 0 for o in out]

TI,TO,VI,VO = splitData(inputs,outputs)

myClass1 = AnnCode(hidden_layers=(1024, 512, 256, 100), activation='relu', max_iter=100, solver='adam', verbose=10,
                   random_state=None, learning_rate_init=.001)

myClass1.fit(np.array(TI), TO)

predicts = myClass1.predict(np.array(VI))
predicts_bin = [1 if p >= 0.5 else 0 for p in predicts]

print("acc: ", accuracy_score(VO, predicts_bin))

Training neural network with 4 hidden layers...
Architecture: 456 -> 1024 -> 512 -> 256 -> 100 -> 1
Epoch 10/100: loss: 0.008829, lr: 0.001000
Epoch 20/100: loss: 0.004013, lr: 0.001000
Epoch 30/100: loss: 0.001264, lr: 0.001000
Epoch 40/100: loss: 0.000338, lr: 0.001000
Epoch 50/100: loss: 0.000036, lr: 0.001000
Epoch 60/100: loss: 0.000041, lr: 0.001000
Epoch 70/100: loss: 0.000011, lr: 0.001000
Epoch 80/100: loss: 0.000007, lr: 0.001000
Epoch 90/100: loss: 0.000001, lr: 0.001000
Epoch 100/100: loss: 0.000000, lr: 0.001000
Training complete. Final loss: 0.000000
acc:  0.5384615384615384


## TF-IDF

In [25]:
inputs = inputTFIDF.copy()
outputs = [ 1 if o == 'positive' else 0 for o in out]

TI,TO,VI,VO = splitData(inputs,outputs)

myClass2 = AnnCode(hidden_layers=(1024,512,256,100), activation='tanh',max_iter=1000,solver='sgd', verbose=10, random_state=None, learning_rate_init=.001)

myClass2.fit(np.array(TI),TO)

predicts = myClass2.predict(np.array(VI))
predicts_bin = [1 if p >= 0.5 else 0 for p in predicts]

print("acc: ", accuracy_score(VO, predicts_bin))

Training neural network with 4 hidden layers...
Architecture: 456 -> 1024 -> 512 -> 256 -> 100 -> 1
Epoch 10/1000: loss: 0.554737, lr: 0.001000
Epoch 20/1000: loss: 0.529864, lr: 0.001000
Epoch 30/1000: loss: 0.506951, lr: 0.001000
Epoch 40/1000: loss: 0.485844, lr: 0.001000
Epoch 50/1000: loss: 0.466402, lr: 0.001000
Epoch 60/1000: loss: 0.448491, lr: 0.001000
Epoch 70/1000: loss: 0.431990, lr: 0.001000
Epoch 80/1000: loss: 0.416786, lr: 0.001000
Epoch 90/1000: loss: 0.402773, lr: 0.001000
Epoch 100/1000: loss: 0.389856, lr: 0.001000
Epoch 110/1000: loss: 0.377945, lr: 0.001000
Epoch 120/1000: loss: 0.366960, lr: 0.001000
Epoch 130/1000: loss: 0.356824, lr: 0.001000
Epoch 140/1000: loss: 0.347469, lr: 0.001000
Epoch 150/1000: loss: 0.338832, lr: 0.001000
Epoch 160/1000: loss: 0.330853, lr: 0.001000
Epoch 170/1000: loss: 0.323480, lr: 0.001000
Epoch 180/1000: loss: 0.316663, lr: 0.001000
Epoch 190/1000: loss: 0.310358, lr: 0.001000
Epoch 200/1000: loss: 0.304522, lr: 0.001000
Epoch 210

# Prediction

In [26]:
text = 'By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.'

cinp = inp.copy()
cinp.append(text)

text_values = Bow.bow_datasets(cinp)[-1]

rez = toolClassifier1.predict([text_values])
print(rez)

rez = myClass1.predict(np.array([text_values]))
print([1 if rez >= 0.5 else 0])

text_values = tfidf.set_tf_idf(cinp)[-1]

rez = toolClassifier2.predict([text_values])
print(rez)

rez = myClass2.predict(np.array([text_values]))
print([1 if rez >= 0.5 else 0])

[1]
[1]
[1]
[1]


# Alte caracteristici: N-grams ( Trigrams )

In [27]:
vectorizer = TfidfVectorizer(ngram_range=(1,3))
inputs = np.array((vectorizer.fit_transform(inp).toarray()))
outputs = [ 1 if o == 'positive' else 0 for o in out] 

TI,TO,VI,VO = splitData(inputs,outputs)

toolClassifier3 = neural_network.MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    max_iter=500,
    solver='adam',
    verbose=10,
    learning_rate_init=0.00005,
)
toolClassifier3.fit(TI, TO)

predicted = toolClassifier3.predict(VI)
print("acc: ", accuracy_score(VO, predicted))


Iteration 1, loss = 0.69552742
Iteration 2, loss = 0.69395875
Iteration 3, loss = 0.69241002
Iteration 4, loss = 0.69089015
Iteration 5, loss = 0.68939472
Iteration 6, loss = 0.68792209
Iteration 7, loss = 0.68646974
Iteration 8, loss = 0.68503441
Iteration 9, loss = 0.68360981
Iteration 10, loss = 0.68219719
Iteration 11, loss = 0.68079657
Iteration 12, loss = 0.67940652
Iteration 13, loss = 0.67802250
Iteration 14, loss = 0.67664785
Iteration 15, loss = 0.67528383
Iteration 16, loss = 0.67392204
Iteration 17, loss = 0.67256280
Iteration 18, loss = 0.67120476
Iteration 19, loss = 0.66984382
Iteration 20, loss = 0.66847753
Iteration 21, loss = 0.66710387
Iteration 22, loss = 0.66571724
Iteration 23, loss = 0.66431368
Iteration 24, loss = 0.66289296
Iteration 25, loss = 0.66145609
Iteration 26, loss = 0.66000175
Iteration 27, loss = 0.65853087
Iteration 28, loss = 0.65703947
Iteration 29, loss = 0.65552594
Iteration 30, loss = 0.65398563
Iteration 31, loss = 0.65241561
Iteration 32, los

In [29]:
myClass3 = AnnCode(hidden_layers=(1024, 512, 256, 100), activation='relu', max_iter=1000, solver='sgd', verbose=10,
                   random_state=None, learning_rate_init=.001)

myClass3.fit(np.array(TI), TO)

predicts = myClass3.predict(np.array(VI))
predicts_bin = [1 if p >= 0.5 else 0 for p in predicts]

print("acc: ", accuracy_score(VO, predicts_bin))

Training neural network with 4 hidden layers...
Architecture: 2593 -> 1024 -> 512 -> 256 -> 100 -> 1
Epoch 10/1000: loss: 0.575903, lr: 0.001000
Epoch 20/1000: loss: 0.563335, lr: 0.001000
Epoch 30/1000: loss: 0.543641, lr: 0.001000
Epoch 40/1000: loss: 0.524368, lr: 0.001000
Epoch 50/1000: loss: 0.506462, lr: 0.001000
Epoch 60/1000: loss: 0.489686, lr: 0.001000
Epoch 70/1000: loss: 0.474033, lr: 0.001000
Epoch 80/1000: loss: 0.459271, lr: 0.001000
Epoch 90/1000: loss: 0.445292, lr: 0.001000
Epoch 100/1000: loss: 0.432159, lr: 0.001000
Epoch 110/1000: loss: 0.419872, lr: 0.001000
Epoch 120/1000: loss: 0.408393, lr: 0.001000
Epoch 130/1000: loss: 0.397585, lr: 0.001000
Epoch 140/1000: loss: 0.387384, lr: 0.001000
Epoch 150/1000: loss: 0.377803, lr: 0.001000
Epoch 160/1000: loss: 0.368818, lr: 0.001000
Epoch 170/1000: loss: 0.360367, lr: 0.001000
Epoch 180/1000: loss: 0.352438, lr: 0.001000
Epoch 190/1000: loss: 0.345005, lr: 0.001000
Epoch 200/1000: loss: 0.338015, lr: 0.001000
Epoch 21

# Predictions

In [34]:
text = 'By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.'

cinp = inp.copy()
cinp.append(text)

vectorizer = TfidfVectorizer(ngram_range=(1,3))
inputs = np.array((vectorizer.fit_transform(inp).toarray()))

rez = toolClassifier3.predict([inputs[-1]])
print(rez)

rez = myClass3.predict(np.array(inputs[-1]))
print([1 if rez >= 0.5 else 0])

[1]
[1]
